# M1 Compact13 Leakage-Safe 안정성 검증

## tl;dr

이 노트북은 09번에서 좋아 보였던 `compact13_overlap` 성능이 전체 CV 중요도를 본 뒤 고른 post-hoc 효과인지 확인한다.
라벨, window, threshold는 고정하고, outer fold의 test fold를 보지 않은 `fold_selected_compact13`을 추가로 비교한다.

## Context & Methods

- 입력: 08번 feature 확장 산출물과 09번 compact feature set 산출물
- main dataset: `strict_no_event20`
- sensitivity dataset: `strict_no_event20_no_event67`
- threshold: `0.6`
- 모델: `DummyClassifier`, `LogisticRegression(class_weight="balanced")`
- 검증: `substation_id` 기준 `StratifiedGroupKFold`

### Key Assumptions

- `compact13_overlap`은 운영 확정이 아니라 다음 실험 후보로만 취급한다.
- 모델 파일 저장과 운영 배포는 하지 않는다.
- 새 threshold 튜닝은 하지 않는다.

In [1]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path
import math
import warnings

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

OUTPUT_DIR = Path("07_데이터산출물")
FEATURES_PATH = OUTPUT_DIR / "m1_feature_expansion_features.csv"
SUMMARY_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
PRIOR_DECISION_PATH = OUTPUT_DIR / "m1_compact_feature_set_decision_matrix.csv"
NOTEBOOK_PATH = Path("06_노트북") / "10_m1_compact13_leakage_safe_validation.ipynb"
REPORT_PATH = OUTPUT_DIR / "10_M1_compact13_leakage_safe_검증_보고서.md"

THRESHOLD = 0.6
RANDOM_STATE = 42
N_SPLITS = 5
POSITIVE_LABEL = "efd_possible"
NEGATIVE_LABEL = "normal"
POSITIVE_VALUE = 1
NEGATIVE_VALUE = 0

OUTPUT_FILES = {
    "feature_selection": OUTPUT_DIR / "m1_compact13_leakage_safe_feature_selection.csv",
    "cv_metrics": OUTPUT_DIR / "m1_compact13_leakage_safe_cv_metrics.csv",
    "cv_predictions": OUTPUT_DIR / "m1_compact13_leakage_safe_cv_predictions.csv",
    "decision_matrix": OUTPUT_DIR / "m1_compact13_leakage_safe_decision_matrix.csv",
    "misclassification_audit": OUTPUT_DIR / "m1_compact13_leakage_safe_misclassification_audit.csv",
    "feature_frequency_png": OUTPUT_DIR / "m1_compact13_selected_feature_frequency.png",
    "misclassification_heatmap_png": OUTPUT_DIR / "m1_compact13_misclassification_heatmap.png",
}

print("입력 경로")
print(f"- features: {FEATURES_PATH}")
print(f"- feature summary: {SUMMARY_PATH}")
print(f"- prior decision matrix: {PRIOR_DECISION_PATH}")

입력 경로
- features: 07_데이터산출물\m1_feature_expansion_features.csv
- feature summary: 07_데이터산출물\m1_compact_feature_set_summary.csv
- prior decision matrix: 07_데이터산출물\m1_compact_feature_set_decision_matrix.csv


## Data

08번 확장 feature CSV를 읽고, 09번 summary에서 `expanded154`와 `compact13_overlap` feature 목록을 그대로 가져온다.
sensitivity dataset은 main dataset에서 Event 67만 제외해 만든다.

In [2]:
def parse_feature_list(value: str) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    return [item.strip() for item in str(value).split("|") if item.strip()]


features_df = pd.read_csv(FEATURES_PATH)
feature_summary_df = pd.read_csv(SUMMARY_PATH)
prior_decision_df = pd.read_csv(PRIOR_DECISION_PATH)

expanded154_features = parse_feature_list(
    feature_summary_df.loc[
        feature_summary_df["feature_set"].eq("expanded154"), "features"
    ].iloc[0]
)
locked_compact13_features = parse_feature_list(
    feature_summary_df.loc[
        feature_summary_df["feature_set"].eq("compact13_overlap"), "features"
    ].iloc[0]
)

missing_expanded = sorted(set(expanded154_features) - set(features_df.columns))
missing_locked = sorted(set(locked_compact13_features) - set(features_df.columns))
assert not missing_expanded, f"expanded154 missing columns: {missing_expanded[:5]}"
assert not missing_locked, f"compact13 missing columns: {missing_locked[:5]}"

features_df["source_event_id"] = features_df["source_event_id"].astype(int)
features_df["event67_flag"] = features_df["event67_flag"].astype(bool)

main_df = features_df.copy()
no_event67_df = features_df.loc[~features_df["event67_flag"]].copy()

datasets = {
    "strict_no_event20": main_df,
    "strict_no_event20_no_event67": no_event67_df,
}

dataset_rows = []
for dataset_id, data in datasets.items():
    fault_events = data.loc[data["source_type"].eq("fault_record"), "source_event_id"].astype(int)
    dataset_rows.append(
        {
            "dataset_id": dataset_id,
            "rows": len(data),
            "normal_rows": int(data["label"].eq(NEGATIVE_LABEL).sum()),
            "positive_rows": int(data["label"].eq(POSITIVE_LABEL).sum()),
            "event67_rows": int(data["event67_flag"].sum()),
            "fault_event20_rows": int(fault_events.eq(20).sum()),
            "fault_event34_rows": int(fault_events.eq(34).sum()),
            "fault_event69_rows": int(fault_events.eq(69).sum()),
            "substation_count": int(data["substation_id"].nunique()),
        }
    )

dataset_summary_df = pd.DataFrame(dataset_rows)
display(dataset_summary_df)

assert dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq("strict_no_event20"), "rows"].iloc[0] == 49
assert dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq("strict_no_event20"), "positive_rows"].iloc[0] == 14
assert dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq("strict_no_event20_no_event67"), "rows"].iloc[0] == 48
assert dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq("strict_no_event20_no_event67"), "positive_rows"].iloc[0] == 13
assert dataset_summary_df[["fault_event20_rows", "fault_event34_rows", "fault_event69_rows"]].sum().sum() == 0
assert dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq("strict_no_event20"), "event67_rows"].iloc[0] == 1
assert dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq("strict_no_event20_no_event67"), "event67_rows"].iloc[0] == 0

print(f"expanded154 feature count: {len(expanded154_features)}")
print(f"locked compact13 feature count: {len(locked_compact13_features)}")
assert len(expanded154_features) == 154
assert len(locked_compact13_features) == 13

,dataset_id,rows,normal_rows,positive_rows,event67_rows,fault_event20_rows,fault_event34_rows,fault_event69_rows,substation_count
0,strict_no_event20,49,35,14,1,0,0,0,29
1,strict_no_event20_no_event67,48,35,13,0,0,0,0,29


expanded154 feature count: 154
locked compact13 feature count: 13


## Model Helpers

`fold_selected_compact13`은 outer fold train subset에서만 expanded154 logistic coefficient를 계산하고, 그 fold의 top13 feature를 선택한다.
test fold는 feature 선택 과정에 사용하지 않는다.

In [3]:
def label_to_binary(labels: pd.Series) -> np.ndarray:
    return labels.eq(POSITIVE_LABEL).astype(int).to_numpy()


def make_logistic_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def make_dummy_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyClassifier(strategy="most_frequent")),
        ]
    )


def get_positive_probability(model: Pipeline, x_frame: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(x_frame)
    classes = list(model.named_steps["model"].classes_)
    if POSITIVE_VALUE not in classes:
        return np.zeros(len(x_frame), dtype=float)
    positive_index = classes.index(POSITIVE_VALUE)
    return probabilities[:, positive_index]


def metric_record(y_true: np.ndarray, y_probability: np.ndarray, threshold: float) -> dict:
    y_pred = (y_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[NEGATIVE_VALUE, POSITIVE_VALUE]
    ).ravel()
    if len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_probability))
    else:
        roc_auc = np.nan
    return {
        "threshold": threshold,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else np.nan,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) else np.nan,
    }


def select_train_only_top13(
    train_frame: pd.DataFrame,
    train_y: np.ndarray,
    candidate_features: list[str],
) -> tuple[list[str], pd.DataFrame]:
    selector = make_logistic_pipeline()
    selector.fit(train_frame[candidate_features], train_y)
    coefficients = selector.named_steps["model"].coef_[0]
    ranking = (
        pd.DataFrame(
            {
                "feature": candidate_features,
                "coefficient": coefficients,
                "abs_coefficient": np.abs(coefficients),
            }
        )
        .sort_values(["abs_coefficient", "feature"], ascending=[False, True])
        .reset_index(drop=True)
    )
    ranking["rank"] = np.arange(1, len(ranking) + 1)
    top20 = ranking.head(20).copy()
    selected = top20.head(13)["feature"].tolist()
    top20["selected_for_model"] = top20["feature"].isin(selected)
    return selected, top20


def mean_pairwise_jaccard(feature_sets: list[set[str]]) -> float:
    pairs = list(combinations(feature_sets, 2))
    if not pairs:
        return np.nan
    scores = []
    for left, right in pairs:
        union = left | right
        scores.append(len(left & right) / len(union) if union else np.nan)
    return float(np.nanmean(scores))


def feature_count_at_least(feature_sets: list[set[str]], minimum_folds: int) -> int:
    counts: dict[str, int] = {}
    for feature_set in feature_sets:
        for feature in feature_set:
            counts[feature] = counts.get(feature, 0) + 1
    return sum(count >= minimum_folds for count in counts.values())

## Run Leakage-Safe CV

같은 outer fold에서 세 전략을 비교한다.

- `expanded154`: 확장 feature 전체
- `locked_compact13`: 09번에서 선택된 13개 feature 고정
- `fold_selected_compact13`: train fold 내부에서만 새로 선택한 13개 feature

In [4]:
strategy_static_features = {
    "expanded154": expanded154_features,
    "locked_compact13": locked_compact13_features,
}
model_factories = {
    "dummy_most_frequent": make_dummy_pipeline,
    "logistic_balanced": make_logistic_pipeline,
}

prediction_records: list[dict] = []
fold_metric_records: list[dict] = []
feature_selection_records: list[dict] = []
fold_selected_sets: dict[tuple[str, str], list[set[str]]] = {}

for dataset_id, data in datasets.items():
    data = data.reset_index(drop=True)
    y = label_to_binary(data["label"])
    groups = data["substation_id"].astype(str).to_numpy()
    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    splits = list(splitter.split(data, y, groups))

    for fold_number, (train_index, test_index) in enumerate(splits, start=1):
        train_frame = data.iloc[train_index].copy()
        test_frame = data.iloc[test_index].copy()
        train_y = y[train_index]
        test_y = y[test_index]
        train_groups = set(train_frame["substation_id"].astype(str))
        test_groups = set(test_frame["substation_id"].astype(str))
        group_overlap = train_groups & test_groups
        assert not group_overlap, f"group overlap in {dataset_id} fold {fold_number}: {group_overlap}"

        selected_top13, top20_ranking = select_train_only_top13(
            train_frame=train_frame,
            train_y=train_y,
            candidate_features=expanded154_features,
        )
        fold_selected_sets.setdefault((dataset_id, "fold_selected_compact13"), []).append(
            set(selected_top13)
        )

        for _, ranking_row in top20_ranking.iterrows():
            feature_selection_records.append(
                {
                    "dataset_id": dataset_id,
                    "fold": fold_number,
                    "selection_strategy": "fold_selected_compact13",
                    "selection_scope": "train_fold_only",
                    "used_test_fold_in_selection": False,
                    "rank": int(ranking_row["rank"]),
                    "feature": ranking_row["feature"],
                    "coefficient": float(ranking_row["coefficient"]),
                    "abs_coefficient": float(ranking_row["abs_coefficient"]),
                    "selected_for_model": bool(ranking_row["selected_for_model"]),
                    "selected_feature_count": len(selected_top13),
                    "train_rows": len(train_frame),
                    "test_rows": len(test_frame),
                    "train_positive_rows": int(train_y.sum()),
                    "test_positive_rows": int(test_y.sum()),
                    "train_groups": "|".join(sorted(map(str, train_groups))),
                    "test_groups": "|".join(sorted(map(str, test_groups))),
                    "group_overlap_count": len(group_overlap),
                }
            )

        strategies = {
            "expanded154": expanded154_features,
            "locked_compact13": locked_compact13_features,
            "fold_selected_compact13": selected_top13,
        }

        for strategy_id, selected_features in strategies.items():
            for model_name, model_factory in model_factories.items():
                model = model_factory()
                model.fit(train_frame[selected_features], train_y)
                test_probability = get_positive_probability(
                    model, test_frame[selected_features]
                )
                test_prediction = (test_probability >= THRESHOLD).astype(int)
                metrics = metric_record(test_y, test_probability, THRESHOLD)
                fold_metric_records.append(
                    {
                        "dataset_id": dataset_id,
                        "feature_strategy": strategy_id,
                        "model": model_name,
                        "fold": fold_number,
                        "feature_count": len(selected_features),
                        "train_rows": len(train_frame),
                        "test_rows": len(test_frame),
                        "train_positive_rows": int(train_y.sum()),
                        "test_positive_rows": int(test_y.sum()),
                        "train_group_count": len(train_groups),
                        "test_group_count": len(test_groups),
                        "group_overlap_count": len(group_overlap),
                        **metrics,
                    }
                )
                for row_position, (_, sample_row) in enumerate(test_frame.iterrows()):
                    prediction_records.append(
                        {
                            "dataset_id": dataset_id,
                            "feature_strategy": strategy_id,
                            "model": model_name,
                            "fold": fold_number,
                            "threshold": THRESHOLD,
                            "feature_count": len(selected_features),
                            "sample_id": sample_row["sample_id"],
                            "source_event_id": int(sample_row["source_event_id"]),
                            "source_type": sample_row["source_type"],
                            "substation_id": int(sample_row["substation_id"]),
                            "label": sample_row["label"],
                            "y_true": int(test_y[row_position]),
                            "y_probability": float(test_probability[row_position]),
                            "y_pred": int(test_prediction[row_position]),
                            "prediction_label": POSITIVE_LABEL
                            if int(test_prediction[row_position]) == POSITIVE_VALUE
                            else NEGATIVE_LABEL,
                            "event67_flag": bool(sample_row["event67_flag"]),
                            "coverage_rate": float(sample_row["coverage_rate"]),
                        }
                    )

feature_selection_df = pd.DataFrame(feature_selection_records)
cv_fold_metrics_df = pd.DataFrame(fold_metric_records)
cv_predictions_df = pd.DataFrame(prediction_records)

mean_metric_columns = [
    "feature_count",
    "train_rows",
    "test_rows",
    "train_positive_rows",
    "test_positive_rows",
    "train_group_count",
    "test_group_count",
    "group_overlap_count",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "false_positive_rate",
    "false_negative_rate",
]
cv_mean_metrics_df = (
    cv_fold_metrics_df.groupby(["dataset_id", "feature_strategy", "model"], as_index=False)[
        mean_metric_columns
    ]
    .mean(numeric_only=True)
    .assign(fold="mean")
)
cv_metrics_df = pd.concat([cv_fold_metrics_df, cv_mean_metrics_df], ignore_index=True, sort=False)

print(f"feature selection rows: {len(feature_selection_df)}")
print(f"cv prediction rows: {len(cv_predictions_df)}")
print(f"cv metric rows: {len(cv_metrics_df)}")
display(cv_mean_metrics_df.sort_values(["dataset_id", "model", "feature_strategy"]))

feature selection rows: 200
cv prediction rows: 582
cv metric rows: 72


,dataset_id,feature_strategy,model,feature_count,train_rows,test_rows,train_positive_rows,test_positive_rows,train_group_count,test_group_count,...,recall,f1,roc_auc,tn,fp,fn,tp,false_positive_rate,false_negative_rate,fold
0,strict_no_event20,expanded154,dummy_most_frequent,154.0,39.2,9.8,11.2,2.8,23.2,5.8,...,0.000000,0.000000,0.500000,7.0,0.0,2.8,0.0,0.000000,1.000000,mean
2,strict_no_event20,fold_selected_compact13,dummy_most_frequent,13.0,39.2,9.8,11.2,2.8,23.2,5.8,...,0.000000,0.000000,0.500000,7.0,0.0,2.8,0.0,0.000000,1.000000,mean
4,strict_no_event20,locked_compact13,dummy_most_frequent,13.0,39.2,9.8,11.2,2.8,23.2,5.8,...,0.000000,0.000000,0.500000,7.0,0.0,2.8,0.0,0.000000,1.000000,mean
1,strict_no_event20,expanded154,logistic_balanced,154.0,39.2,9.8,11.2,2.8,23.2,5.8,...,0.500000,0.452381,0.712302,5.6,1.4,1.4,1.4,0.210714,0.500000,mean
3,strict_no_event20,fold_selected_compact13,logistic_balanced,13.0,39.2,9.8,11.2,2.8,23.2,5.8,...,0.600000,0.500000,0.715873,5.8,1.2,1.2,1.6,0.177381,0.400000,mean
5,strict_no_event20,locked_compact13,logistic_balanced,13.0,39.2,9.8,11.2,2.8,23.2,5.8,...,0.733333,0.740000,0.911111,6.6,0.4,0.8,2.0,0.061905,0.266667,mean
6,strict_no_event20_no_event67,expanded154,dummy_most_frequent,154.0,38.4,9.6,10.4,2.6,23.2,5.8,...,0.000000,0.000000,0.500000,7.0,0.0,2.6,0.0,0.000000,1.000000,mean
8,strict_no_event20_no_event67,fold_selected_compact13,dummy_most_frequent,13.0,38.4,9.6,10.4,2.6,23.2,5.8,...,0.000000,0.000000,0.500000,7.0,0.0,2.6,0.0,0.000000,1.000000,mean
10,strict_no_event20_no_event67,locked_compact13,dummy_most_frequent,13.0,38.4,9.6,10.4,2.6,23.2,5.8,...,0.000000,0.000000,0.500000,7.0,0.0,2.6,0.0,0.000000,1.000000,mean
7,strict_no_event20_no_event67,expanded154,logistic_balanced,154.0,38.4,9.6,10.4,2.6,23.2,5.8,...,0.433333,0.384762,0.713492,6.0,1.0,1.4,1.2,0.147619,0.566667,mean


## Decision Matrix

의사결정은 `LogisticRegression`의 out-of-fold pooled prediction을 threshold 0.6으로 평가한다.
`fold_selected_compact13`은 각 fold의 train-only feature selection 결과를 사용한다.

In [5]:
def pooled_strategy_metrics(predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    logistic_predictions = predictions.loc[predictions["model"].eq("logistic_balanced")].copy()
    for (dataset_id, strategy_id), group in logistic_predictions.groupby(
        ["dataset_id", "feature_strategy"], sort=True
    ):
        y_true = group["y_true"].to_numpy()
        y_probability = group["y_probability"].to_numpy()
        metrics = metric_record(y_true, y_probability, THRESHOLD)
        rows.append(
            {
                "dataset_id": dataset_id,
                "feature_strategy": strategy_id,
                "threshold": THRESHOLD,
                "feature_count": int(group["feature_count"].iloc[0])
                if strategy_id != "fold_selected_compact13"
                else int(round(group["feature_count"].mean())),
                "rows": len(group),
                "normal_rows": int((group["y_true"] == NEGATIVE_VALUE).sum()),
                "positive_rows": int((group["y_true"] == POSITIVE_VALUE).sum()),
                **metrics,
            }
        )
    return pd.DataFrame(rows)


decision_matrix_df = pooled_strategy_metrics(cv_predictions_df)

stability_rows = []
for dataset_id in datasets:
    selected_sets = fold_selected_sets[(dataset_id, "fold_selected_compact13")]
    feature_frequency = {}
    for selected_set in selected_sets:
        for feature in selected_set:
            feature_frequency[feature] = feature_frequency.get(feature, 0) + 1
    stability_rows.append(
        {
            "dataset_id": dataset_id,
            "feature_strategy": "fold_selected_compact13",
            "mean_pairwise_jaccard": mean_pairwise_jaccard(selected_sets),
            "features_selected_in_at_least_3_folds": feature_count_at_least(
                selected_sets, minimum_folds=3
            ),
            "unique_selected_feature_count": len(feature_frequency),
            "max_feature_frequency": max(feature_frequency.values()) if feature_frequency else 0,
        }
    )
stability_df = pd.DataFrame(stability_rows)

decision_matrix_df = decision_matrix_df.merge(
    stability_df,
    on=["dataset_id", "feature_strategy"],
    how="left",
)
decision_matrix_df[[
    "mean_pairwise_jaccard",
    "features_selected_in_at_least_3_folds",
    "unique_selected_feature_count",
    "max_feature_frequency",
]] = decision_matrix_df[[
    "mean_pairwise_jaccard",
    "features_selected_in_at_least_3_folds",
    "unique_selected_feature_count",
    "max_feature_frequency",
]].fillna({
    "mean_pairwise_jaccard": np.nan,
    "features_selected_in_at_least_3_folds": np.nan,
    "unique_selected_feature_count": np.nan,
    "max_feature_frequency": np.nan,
})

expanded_baseline = (
    decision_matrix_df.loc[decision_matrix_df["feature_strategy"].eq("expanded154")]
    .set_index("dataset_id")
    .to_dict("index")
)
comparison_rows = []
for _, row in decision_matrix_df.iterrows():
    baseline = expanded_baseline[row["dataset_id"]]
    ba_delta = row["balanced_accuracy"] - baseline["balanced_accuracy"]
    recall_delta = row["recall"] - baseline["recall"]
    fpr_delta = row["false_positive_rate"] - baseline["false_positive_rate"]
    ba_pass = ba_delta >= -0.03
    recall_pass = recall_delta >= -0.05
    fpr_pass = fpr_delta <= 0.05
    if row["feature_strategy"] == "fold_selected_compact13":
        stability_pass = (
            row["mean_pairwise_jaccard"] >= 0.20
            and row["features_selected_in_at_least_3_folds"] >= 4
        )
        candidate_pass = bool(ba_pass and recall_pass and fpr_pass and stability_pass)
    else:
        stability_pass = np.nan
        candidate_pass = np.nan
    comparison_rows.append(
        {
            "ba_delta_vs_expanded154": float(ba_delta),
            "recall_delta_vs_expanded154": float(recall_delta),
            "fpr_delta_vs_expanded154": float(fpr_delta),
            "ba_pass": ba_pass if row["feature_strategy"] == "fold_selected_compact13" else np.nan,
            "recall_pass": recall_pass if row["feature_strategy"] == "fold_selected_compact13" else np.nan,
            "fpr_pass": fpr_pass if row["feature_strategy"] == "fold_selected_compact13" else np.nan,
            "stability_pass": stability_pass,
            "candidate_pass": candidate_pass,
        }
    )
decision_matrix_df = pd.concat(
    [decision_matrix_df.reset_index(drop=True), pd.DataFrame(comparison_rows)], axis=1
)

fold_selected_rows = decision_matrix_df.loc[
    decision_matrix_df["feature_strategy"].eq("fold_selected_compact13")
]
if bool(fold_selected_rows["candidate_pass"].fillna(False).all()):
    final_decision = "compact13을 다음 학습 기준으로 유지"
elif (decision_matrix_df.loc[decision_matrix_df["feature_strategy"].eq("expanded154"), "balanced_accuracy"] >= 0.60).all():
    final_decision = "compact13은 post-hoc 영향 가능성이 있어 expanded154 유지"
else:
    final_decision = "둘 다 불안정하므로 normal negative 재설계 우선"

decision_matrix_df["final_decision"] = final_decision
display(
    decision_matrix_df.sort_values(["dataset_id", "feature_strategy"])[
        [
            "dataset_id",
            "feature_strategy",
            "feature_count",
            "balanced_accuracy",
            "recall",
            "false_positive_rate",
            "fp",
            "fn",
            "ba_delta_vs_expanded154",
            "recall_delta_vs_expanded154",
            "fpr_delta_vs_expanded154",
            "mean_pairwise_jaccard",
            "features_selected_in_at_least_3_folds",
            "candidate_pass",
        ]
    ]
)
print(final_decision)

,dataset_id,feature_strategy,feature_count,balanced_accuracy,recall,false_positive_rate,fp,fn,ba_delta_vs_expanded154,recall_delta_vs_expanded154,fpr_delta_vs_expanded154,mean_pairwise_jaccard,features_selected_in_at_least_3_folds,candidate_pass
0,strict_no_event20,expanded154,154,0.650000,0.500000,0.200000,7,7,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,strict_no_event20,fold_selected_compact13,13,0.700000,0.571429,0.171429,6,6,0.050000,0.071429,-0.028571,0.340555,9.0,True
2,strict_no_event20,locked_compact13,13,0.828571,0.714286,0.057143,2,4,0.178571,0.214286,-0.142857,NaN,NaN,NaN
3,strict_no_event20_no_event67,expanded154,154,0.659341,0.461538,0.142857,5,7,0.000000,0.000000,0.000000,NaN,NaN,NaN
4,strict_no_event20_no_event67,fold_selected_compact13,13,0.764835,0.615385,0.085714,3,5,0.105495,0.153846,-0.057143,0.398048,10.0,True
5,strict_no_event20_no_event67,locked_compact13,13,0.817582,0.692308,0.057143,2,4,0.158242,0.230769,-0.085714,NaN,NaN,NaN


compact13을 다음 학습 기준으로 유지


## Misclassification Audit And Visuals

FP/FN 이벤트를 audit CSV로 남기고, fold-selected feature 빈도와 오분류 패턴을 PNG로 저장한다.

In [6]:
logistic_predictions_df = cv_predictions_df.loc[
    cv_predictions_df["model"].eq("logistic_balanced")
].copy()
misclassification_audit_df = logistic_predictions_df.loc[
    logistic_predictions_df["y_true"].ne(logistic_predictions_df["y_pred"])
].copy()
misclassification_audit_df["error_type"] = np.where(
    (misclassification_audit_df["y_true"].eq(NEGATIVE_VALUE))
    & (misclassification_audit_df["y_pred"].eq(POSITIVE_VALUE)),
    "FP",
    "FN",
)
misclassification_audit_df = misclassification_audit_df[
    [
        "dataset_id",
        "feature_strategy",
        "fold",
        "error_type",
        "sample_id",
        "source_event_id",
        "source_type",
        "substation_id",
        "label",
        "y_probability",
        "prediction_label",
        "event67_flag",
        "coverage_rate",
    ]
].sort_values(["dataset_id", "feature_strategy", "error_type", "source_event_id"])

selected_frequency_df = (
    feature_selection_df.loc[feature_selection_df["selected_for_model"]]
    .groupby(["dataset_id", "feature"], as_index=False)
    .size()
    .rename(columns={"size": "selected_fold_count"})
)
selected_frequency_df["feature_short"] = selected_frequency_df["feature"].str.replace(
    "__", "\n", regex=False
)

frequency_pivot = selected_frequency_df.pivot_table(
    index="feature",
    columns="dataset_id",
    values="selected_fold_count",
    fill_value=0,
    aggfunc="sum",
)
frequency_pivot["total"] = frequency_pivot.sum(axis=1)
frequency_pivot = frequency_pivot.sort_values("total", ascending=False).head(25)

plt.figure(figsize=(12, 8))
x = np.arange(len(frequency_pivot.index))
width = 0.38
dataset_ids = [dataset_id for dataset_id in datasets.keys()]
for offset, dataset_id in enumerate(dataset_ids):
    values = frequency_pivot.get(dataset_id, pd.Series(0, index=frequency_pivot.index))
    plt.bar(x + (offset - 0.5) * width, values, width=width, label=dataset_id)
plt.xticks(x, [feature.replace("__", "\n") for feature in frequency_pivot.index], rotation=70, ha="right")
plt.ylabel("selected folds")
plt.title("Fold-selected compact13 feature frequency")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FILES["feature_frequency_png"], dpi=160)
plt.close()

heatmap_counts = (
    misclassification_audit_df.groupby(["dataset_id", "feature_strategy", "error_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for error_type in ["FP", "FN"]:
    if error_type not in heatmap_counts.columns:
        heatmap_counts[error_type] = 0
heatmap_counts["total_errors"] = heatmap_counts["FP"] + heatmap_counts["FN"]
heatmap_pivot = heatmap_counts.pivot_table(
    index="feature_strategy",
    columns="dataset_id",
    values="total_errors",
    fill_value=0,
    aggfunc="sum",
)
plt.figure(figsize=(8, 4.8))
image = plt.imshow(heatmap_pivot.values, aspect="auto", cmap="Blues")
plt.colorbar(image, label="FP + FN")
plt.xticks(np.arange(len(heatmap_pivot.columns)), heatmap_pivot.columns, rotation=20, ha="right")
plt.yticks(np.arange(len(heatmap_pivot.index)), heatmap_pivot.index)
for row_idx in range(heatmap_pivot.shape[0]):
    for col_idx in range(heatmap_pivot.shape[1]):
        plt.text(
            col_idx,
            row_idx,
            int(heatmap_pivot.values[row_idx, col_idx]),
            ha="center",
            va="center",
            color="black",
        )
plt.title("Misclassification count heatmap at threshold 0.6")
plt.tight_layout()
plt.savefig(OUTPUT_FILES["misclassification_heatmap_png"], dpi=160)
plt.close()

display(misclassification_audit_df.head(20))
print(f"misclassification rows: {len(misclassification_audit_df)}")
print(f"feature frequency image: {OUTPUT_FILES['feature_frequency_png']}")
print(f"misclassification image: {OUTPUT_FILES['misclassification_heatmap_png']}")

,dataset_id,feature_strategy,fold,error_type,sample_id,source_event_id,source_type,substation_id,label,y_probability,prediction_label,event67_flag,coverage_rate
201,strict_no_event20,expanded154,4,FN,strict_positive_0003,3,fault_record,12,efd_possible,0.004764,normal,False,1.00000
202,strict_no_event20,expanded154,4,FN,strict_positive_0015,15,fault_record,29,efd_possible,0.264783,normal,False,1.00000
203,strict_no_event20,expanded154,4,FN,strict_positive_0040,40,fault_record,24,efd_possible,0.011252,normal,False,0.99504
85,strict_no_event20,expanded154,2,FN,strict_positive_0052,52,fault_record,21,efd_possible,0.008455,normal,False,1.00000
149,strict_no_event20,expanded154,3,FN,strict_positive_0053,53,fault_record,13,efd_possible,0.252051,normal,False,1.00000
257,strict_no_event20,expanded154,5,FN,strict_positive_0064,64,fault_record,6,efd_possible,0.541464,normal,False,1.00000
87,strict_no_event20,expanded154,2,FN,strict_positive_0067,67,fault_record,7,efd_possible,0.051457,normal,True,1.00000
249,strict_no_event20,expanded154,5,FP,normal_0008,8,normal_event,6,normal,0.972570,efd_possible,False,1.00000
14,strict_no_event20,expanded154,1,FP,normal_0019,19,normal_event,8,normal,0.905383,efd_possible,False,1.00000
196,strict_no_event20,expanded154,4,FP,normal_0021,21,normal_event,1,normal,0.699078,efd_possible,False,1.00000


misclassification rows: 58
feature frequency image: 07_데이터산출물\m1_compact13_selected_feature_frequency.png
misclassification image: 07_데이터산출물\m1_compact13_misclassification_heatmap.png


## Save Outputs

CSV와 PNG, 보고서를 저장한다.

In [7]:
feature_selection_df.to_csv(OUTPUT_FILES["feature_selection"], index=False, encoding="utf-8-sig")
cv_metrics_df.to_csv(OUTPUT_FILES["cv_metrics"], index=False, encoding="utf-8-sig")
cv_predictions_df.to_csv(OUTPUT_FILES["cv_predictions"], index=False, encoding="utf-8-sig")
decision_matrix_df.to_csv(OUTPUT_FILES["decision_matrix"], index=False, encoding="utf-8-sig")
misclassification_audit_df.to_csv(
    OUTPUT_FILES["misclassification_audit"], index=False, encoding="utf-8-sig"
)

def markdown_table(frame: pd.DataFrame, float_digits: int = 4) -> str:
    display_frame = frame.copy()
    for column in display_frame.columns:
        if pd.api.types.is_float_dtype(display_frame[column]):
            display_frame[column] = display_frame[column].map(
                lambda value: "" if pd.isna(value) else f"{value:.{float_digits}f}"
            )
    display_frame = display_frame.fillna("")

    def format_cell(value) -> str:
        text = str(value)
        return text.replace("|", "\|").replace("\n", "<br>")

    headers = [format_cell(column) for column in display_frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in display_frame.itertuples(index=False):
        lines.append("| " + " | ".join(format_cell(value) for value in row) + " |")
    return "\n".join(lines)


report_decision_columns = [
    "dataset_id",
    "feature_strategy",
    "feature_count",
    "balanced_accuracy",
    "recall",
    "f1",
    "false_positive_rate",
    "fp",
    "fn",
    "ba_delta_vs_expanded154",
    "recall_delta_vs_expanded154",
    "fpr_delta_vs_expanded154",
    "mean_pairwise_jaccard",
    "features_selected_in_at_least_3_folds",
    "candidate_pass",
]
report_decision_df = decision_matrix_df[report_decision_columns].sort_values(
    ["dataset_id", "feature_strategy"]
)

report_mean_metrics_df = cv_mean_metrics_df.loc[
    cv_mean_metrics_df["model"].eq("logistic_balanced"),
    [
        "dataset_id",
        "feature_strategy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "fp",
        "fn",
    ],
].sort_values(["dataset_id", "feature_strategy"])

feature_stability_summary_df = stability_df.copy()
selected_frequency_summary = (
    selected_frequency_df.sort_values(["selected_fold_count", "feature"], ascending=[False, True])
    .groupby("dataset_id")
    .head(10)
    .sort_values(["dataset_id", "selected_fold_count", "feature"], ascending=[True, False, True])
)

fp_fn_summary_df = (
    misclassification_audit_df.groupby(["dataset_id", "feature_strategy", "error_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for error_type in ["FP", "FN"]:
    if error_type not in fp_fn_summary_df.columns:
        fp_fn_summary_df[error_type] = 0
fp_fn_summary_df = fp_fn_summary_df[["dataset_id", "feature_strategy", "FP", "FN"]]

if final_decision == "compact13을 다음 학습 기준으로 유지":
    decision_explanation = (
        "fold별 train-only feature selection에서도 expanded154 대비 성능 하락 기준을 넘지 않았고, "
        "오탐률도 악화되지 않았다. 따라서 compact13 계열을 다음 학습 기준 후보로 유지한다."
    )
elif final_decision == "compact13은 post-hoc 영향 가능성이 있어 expanded154 유지":
    decision_explanation = (
        "locked compact13은 좋아 보일 수 있으나 fold-selected 검증에서 기준을 충분히 통과하지 못했다. "
        "따라서 feature 축소 확정은 보류하고 expanded154를 유지한다."
    )
else:
    decision_explanation = (
        "compact 계열과 expanded154 모두 다음 기준으로 잠그기 어렵다. "
        "feature 추가보다 normal negative 재설계가 먼저다."
    )

dataset_table_rows = [
    "| dataset_id | rows | normal_rows | positive_rows | event67_rows | fault_event20_rows | fault_event34_rows | fault_event69_rows | substation_count |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for row in dataset_summary_df.itertuples(index=False):
    dataset_table_rows.append(
        f"| {row.dataset_id} | {row.rows} | {row.normal_rows} | {row.positive_rows} | "
        f"{row.event67_rows} | {row.fault_event20_rows} | {row.fault_event34_rows} | "
        f"{row.fault_event69_rows} | {row.substation_count} |"
    )

report_parts = [
    "# M1 compact13 leakage-safe 검증 보고서",
    "",
    "## 결론",
    "",
    f"- 최종 판단: {final_decision}",
    "- 기준 라벨/window/threshold는 유지했다.",
    "- main dataset은 `strict_no_event20`, sensitivity dataset은 `strict_no_event20_no_event67`이다.",
    "- threshold는 0.6만 의사결정 기준으로 사용했다.",
    f"- 핵심 해석: {decision_explanation}",
    "",
    "## 검증 질문",
    "",
    "09번의 `compact13_overlap`은 성능이 가장 좋아 보였지만, 08번 전체 CV 중요도를 본 뒤 feature를 고른 post-hoc feature selection 결과였다.",
    "이번 검증은 outer fold의 test fold를 feature 선택에 전혀 쓰지 않고도 compact13 계열 성능이 유지되는지 확인한다.",
    "",
    "## 입력과 Dataset",
    "",
    "\n".join(dataset_table_rows),
    "",
    "## 방법",
    "",
    "- `expanded154`: 08번의 확장 feature 전체 154개를 그대로 사용했다.",
    "- `locked_compact13`: 09번에서 선택된 13개 feature를 고정해 재평가했다.",
    "- `fold_selected_compact13`: 각 outer fold에서 train fold 안에서만 logistic coefficient importance를 계산하고 top13 feature를 선택했다.",
    "- 모든 모델 평가는 `substation_id` 기준 group CV out-of-fold prediction으로 했다.",
    "- metadata, 날짜, event id, `substation_id`, coverage 계열은 학습 입력으로 쓰지 않았다.",
    "",
    "## Threshold 0.6 Decision Matrix",
    "",
    markdown_table(report_decision_df),
    "",
    "## Group CV 평균 성능",
    "",
    markdown_table(report_mean_metrics_df),
    "",
    "## Fold별 Feature 선택 안정성",
    "",
    markdown_table(feature_stability_summary_df),
    "",
    "## Fold-selected 상위 선택 빈도",
    "",
    markdown_table(selected_frequency_summary[["dataset_id", "feature", "selected_fold_count"]]),
    "",
    "## 오분류 요약",
    "",
    markdown_table(fp_fn_summary_df),
    "",
    "## 생성 이미지",
    "",
    "- `07_데이터산출물/m1_compact13_selected_feature_frequency.png`",
    "- `07_데이터산출물/m1_compact13_misclassification_heatmap.png`",
    "",
    "## 검증 결과",
    "",
    "- Event 20/34/69는 학습 dataset에 없다.",
    "- Event 67은 `strict_no_event20`에만 있고 `strict_no_event20_no_event67`에는 없다.",
    "- group CV에서 같은 `substation_id`가 train/test에 동시에 들어가지 않았다.",
    "- `fold_selected_compact13`의 feature selection은 `train_fold_only`로 기록했고, `used_test_fold_in_selection=False`로 검증했다.",
    "- misclassification audit에는 threshold 0.6 기준 FP/FN 이벤트를 기록했다.",
    "",
    "## 한계와 주의점",
    "",
    "- 샘플 수가 여전히 작기 때문에 이번 결과도 운영 모델 확정 근거가 아니라 다음 학습 기준 후보를 고르는 근거다.",
    "- `fold_selected_compact13`은 test fold 누수는 막았지만, feature 선택 규칙 자체는 아직 logistic coefficient에 의존한다.",
    "- 모델 파일 저장이나 운영 배포는 하지 않았다.",
    "",
    "## 다음 단계",
    "",
    "- 이번 기준이 통과하면 `compact13 + threshold 0.6`으로 모델 해석과 오분류 이벤트 리뷰를 진행한다.",
    "- 기준이 통과하지 못하면 compact 확정은 보류하고 expanded154 유지 또는 normal negative 재설계를 먼저 검토한다.",
    "",
]
report = "\n".join(report_parts)
REPORT_PATH.write_text(report, encoding="utf-8")

print(f"saved report: {REPORT_PATH}")
for name, path in OUTPUT_FILES.items():
    print(f"saved {name}: {path}")

saved report: 07_데이터산출물\10_M1_compact13_leakage_safe_검증_보고서.md
saved feature_selection: 07_데이터산출물\m1_compact13_leakage_safe_feature_selection.csv
saved cv_metrics: 07_데이터산출물\m1_compact13_leakage_safe_cv_metrics.csv
saved cv_predictions: 07_데이터산출물\m1_compact13_leakage_safe_cv_predictions.csv
saved decision_matrix: 07_데이터산출물\m1_compact13_leakage_safe_decision_matrix.csv
saved misclassification_audit: 07_데이터산출물\m1_compact13_leakage_safe_misclassification_audit.csv
saved feature_frequency_png: 07_데이터산출물\m1_compact13_selected_feature_frequency.png
saved misclassification_heatmap_png: 07_데이터산출물\m1_compact13_misclassification_heatmap.png


<>:20: SyntaxWarning: invalid escape sequence '\|'
<>:20: SyntaxWarning: invalid escape sequence '\|'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30664\993086562.py:20: SyntaxWarning: invalid escape sequence '\|'
  return text.replace("|", "\|").replace("\n", "<br>")


## Validation

산출물이 계획 조건을 만족하는지 실행 결과로 확인한다.

In [8]:
for name, path in OUTPUT_FILES.items():
    assert path.exists(), f"missing output: {name} -> {path}"
assert REPORT_PATH.exists(), f"missing report: {REPORT_PATH}"

saved_feature_selection = pd.read_csv(OUTPUT_FILES["feature_selection"])
saved_predictions = pd.read_csv(OUTPUT_FILES["cv_predictions"])
saved_decision = pd.read_csv(OUTPUT_FILES["decision_matrix"])
saved_misclassification = pd.read_csv(OUTPUT_FILES["misclassification_audit"])

assert saved_feature_selection["group_overlap_count"].max() == 0
assert saved_predictions.loc[saved_predictions["dataset_id"].eq("strict_no_event20"), "source_event_id"].eq(67).any()
assert not saved_predictions.loc[
    saved_predictions["dataset_id"].eq("strict_no_event20_no_event67"), "source_event_id"
].eq(67).any()
learning_fault_events = features_df.loc[features_df["source_type"].eq("fault_record"), "source_event_id"].astype(int)
assert not learning_fault_events.isin([20, 34, 69]).any()
assert saved_feature_selection["selection_scope"].eq("train_fold_only").all()
assert not saved_feature_selection["used_test_fold_in_selection"].astype(bool).any()
assert set(saved_decision["threshold"].round(4)) == {THRESHOLD}
assert {"FP", "FN"}.intersection(set(saved_misclassification["error_type"]))

for png_path in [
    OUTPUT_FILES["feature_frequency_png"],
    OUTPUT_FILES["misclassification_heatmap_png"],
]:
    image_array = mpimg.imread(png_path)
    assert image_array.size > 0, f"blank image: {png_path}"
    assert float(np.nanstd(image_array)) > 0, f"flat image: {png_path}"

text_outputs = [
    REPORT_PATH,
    OUTPUT_FILES["feature_selection"],
    OUTPUT_FILES["cv_metrics"],
    OUTPUT_FILES["cv_predictions"],
    OUTPUT_FILES["decision_matrix"],
    OUTPUT_FILES["misclassification_audit"],
]
forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
for path in text_outputs:
    text = path.read_text(encoding="utf-8-sig" if path.suffix == ".csv" else "utf-8")
    for term in forbidden_terms:
        assert term not in text, f"forbidden term {term!r} found in {path}"

print("검증 완료")
print(f"- feature selection rows: {len(saved_feature_selection)}")
print(f"- prediction rows: {len(saved_predictions)}")
print(f"- decision rows: {len(saved_decision)}")
print(f"- misclassification rows: {len(saved_misclassification)}")
print(f"- final decision: {saved_decision['final_decision'].iloc[0]}")

검증 완료
- feature selection rows: 200
- prediction rows: 582
- decision rows: 6
- misclassification rows: 58
- final decision: compact13을 다음 학습 기준으로 유지
